# 00 — Affinity Data Release - SensoFit Walkthrough

This notebook accompanies the affinity data release distributed via Fragalysis.
It walks through, end-to-end, and illustrates **the fitting apporach behing the
kinetic parameters** from the raw GCI sensorgrams, and explores the
methodological choices and current challenges we faced along the way.

**Contents**

1. [Loading the data package](#2-loading-the-data-package)
2. [Worked example: DK + ODE fit of a single sensorgram](#3-worked-example-dk--ode-fit-of-a-single-sensorgram)
3. [Non-convex optimisation: DK seeds vs random starting points](#4-non-convex-optimisation-dk-seeds-vs-random-starting-points)
4. [Residual weighting and goodness-of-fit](#5-residual-weighting-and-goodness-of-fit)
5. [The case of negative steady-state values after last dissociation](#6-the-case-of-negative-steady-state-values-after-last-dissociation)
6. [Brief comparison with Creoptix evaluations](#7-brief-comparison-with-creoptix-evaluations)

The notebook is self-contained: every helper function used in this analysis is
defined inline, so you can read it top-to-bottom without having to jump into
the `sensofit` source tree.

## 1. Loading the data package

Below is a schematic representation of the layout of the data package produced by
`sensofit.dataexporter.export_package`:

```
{package}.zip
├── README.md
├── all_creoptix_kinetics_evaluations.csv
└── {experiment}/
    ├── experiment.json
    ├── creoptix_kinetics_evaluations.csv
    └── {compound}__{conc}__cyc{idx:03d}/
        ├── metadata.json
        ├── kinetics.json
        ├── FC2-FC1.csv      # time_s, signal, raw_active, raw_reference
        └── FC3-FC1.csv
```

`sensofit.load_experiment(path)` is a dispatcher that accepts:

- a `.cxw` file              → reads via `sensofit.load_cxw`
- a `.zip` data package      → reads via `sensofit.load_package`
- an unzipped package folder → reads via `sensofit.load_package`

The returned dict has identical shape in all three cases, so the rest of this
notebook (and indeed the rest of the `sensofit` API) works unchanged.

This section will show you how to extract and visualise the double referenced sensorgram
for one sample from one channel of one of the four experiments.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.stats import gaussian_kde

from sensofit import load_experiment
from sensofit.models import (
    build_concentration_profile,
    build_pulsed_concentration_profile,
    select_dmso_cal,
    double_reference,
    build_full_weight_mask,
    trim_to_fit_window,
    simulate_sensorgram,
)
from sensofit.direct_kinetics import fit_sample as dk_fit_sample
from sensofit.ode_fitting import fit_sample as ode_fit_sample
from sensofit.ode_fitting import _residuals_full

In [ ]:
# --- Reusable loader cell --------------------------------------------------
# Edit INPUT to point at the data package you downloaded from Fragalysis.
INPUT = "/mnt/c/Users/kgr26424/Documents/Creoptix_test/Data_release/Test/data-release-V1.zip"  # raw .cxw file
# name of the experiment to load from the package, choose from: 20260318_EV71_2A_Binding_assay, 20260323_EV71_2A_Binding_assay, 20260326_EV71_2A_Binding_assay, 20260330_EV71_2A_Binding_assay
EXPERIMENT="20260323_EV71_2A_Binding_assay" 

data = load_experiment(INPUT, name=EXPERIMENT)
samples      = data['samples']
blanks       = data['blanks']
dmso_cals    = data['dmso_cals']
other_cycles = data.get('other_cycles', [])

print(f'Source:        {INPUT}')
print(f'Samples:       {len(samples)}')
print(f'Blanks:        {len(blanks)}')
print(f'DMSO Cals:     {len(dmso_cals)}')
print(f'Other cycles:  {len(other_cycles)}')

In [ ]:
# Quick inventory of the first 10 sample cycles
inv = pd.DataFrame([
    {'cycle_index': s['index'],
     'compound': s['compound'],
     'concentration_uM': s['concentration_M'] * 1e6,
     'mw': s.get('mw', 0)}
    for s in samples
])
inv.head(10)

In [ ]:
# Pick the featured sample ID used throughout this walkthrough using its ASAP ID or "control".
sample_id = "ASAP-0044506"
sample = next(s for s in samples if sample_id in s['compound'])
print(f'Featured sample: {sample["compound"]}  '
      f'({sample["concentration_M"]*1e6:.1f} µM, cycle {sample["index"]})')

In [ ]:
# Helper: yield contiguous (start, end) time spans where mask is True.
def _contiguous_spans(t, mask):
    spans, in_span, start = [], False, None
    for i, v in enumerate(mask):
        if v and not in_span:
            start, in_span = t[i], True
        elif not v and in_span:
            spans.append((start, t[i - 1])); in_span = False
    if in_span:
        spans.append((start, t[-1]))
    return spans

In [ ]:
# --- Section 1 plot: double-referenced sensorgram with shaded phases ------
# Baseline (white)  →  Association: analyte pulses (orange) + buffer pulses
# (blue, the regions sensofit fits during association)  →  Dissociation
# (green, fitted from Rinse → RinseEnd).
t_raw   = sample['time']
sig_dr, blank_idx = double_reference(sample, blanks)
inj   = sample['markers'].get('Injection', t_raw[0])
rinse = sample['markers'].get('Rinse',     t_raw[-1])
rend  = sample['markers'].get('RinseEnd',  t_raw[-1])

dmso_cal = select_dmso_cal(sample['index'], dmso_cals)
w_full   = build_full_weight_mask(t_raw, sample['markers'], dmso_cal,
                                  association_weight=0.0)
buf_mask = (w_full > 0) & (t_raw >= inj) & (t_raw < rinse)

fig, ax = plt.subplots(figsize=(12, 4.5))
# Phase shading
ax.axvspan(t_raw[0], inj,   alpha=0.08, color='gray',   label='Baseline (not fitted)')
ax.axvspan(inj,      rinse, alpha=0.10, color='orange', label='Association — analyte pulses (RI)')
for s_, e_ in _contiguous_spans(t_raw, buf_mask):
    ax.axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
ax.axvspan(rinse,    rend,  alpha=0.18, color='mediumseagreen',
           label='Dissociation (fitted)')
ax.fill_between([], [], alpha=0.18, color='cornflowerblue',
                label='Association — buffer pulses (fitted)')

ax.plot(t_raw, sig_dr, 'k-', lw=0.6, label=f'Double-referenced data (blank {blank_idx})')
ax.axhline(0, color='gray', ls=':', lw=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Signal (pg/mm²)')
ax.set_title(f'{sample["compound"]} — {sample["concentration_M"]*1e6:.1f} µM '
             f'(cycle {sample["index"]})')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Worked example: DK + ODE fit of a single sensorgram

`sensofit` fits each sensorgram with a two-stage pipeline:

1. **Direct Kinetics (DK)** — closed-form linear solver on the *envelope*
   concentration profile $c_\text{env}(t)$. Cheap, deterministic, and gives
   excellent initial guesses of $(k_a, k_d, R_{\max})$.
2. **ODE refinement** — non-linear least-squares fit of the 1:1 Langmuir ODE
   on the *pulsed* concentration profile $c_\text{pulsed}(t)$, weighted on
   the buffer-pulse intervals during association plus the dissociation
   tail. Multi-start TRF, seeded from DK.

This section walks you through how to run both stages on the featured sample
and overlay the results of both approaches as well as the WAVEcontrol fit from
the Creoptix software.

In [ ]:
n_starts = 5  # ODE fitting is non-convex, so we run it from multiple starting points to find a good fit.
dk  = dk_fit_sample(sample, dmso_cals, blanks=blanks)
ode = ode_fit_sample(sample, dmso_cals, blanks=blanks, n_starts=n_starts)

print('=== Direct Kinetics ===')
print(f'  ka   = {dk["ka"]:.2e} M⁻¹s⁻¹')
print(f'  kd   = {dk["kd"]:.4f} s⁻¹')
print(f'  Rmax = {dk["Rmax"]:.1f} pg/mm²')
print(f'  KD   = {dk["KD"]*1e6:.2f} µM')
print()
print(f'=== ODE refinement ({n_starts} starts) ===')
print(f'  ka   = {ode["ka"]:.2e} ± {ode.get("ka_se", float("nan")):.2e}')
print(f'  kd   = {ode["kd"]:.4f}  ± {ode.get("kd_se", float("nan")):.4f}')
print(f'  Rmax = {ode["Rmax"]:.1f}   ± {ode.get("Rmax_se", float("nan")):.2f}')
print(f'  KD   = {ode["KD"]*1e6:.2f} µM')
print(f'  σ    = {ode["sigma_residual"]:.3f} pg/mm²   ({ode["message"]})')

In [ ]:
# Overlay ODE fit on the data, with the same phase shading as Section 1.
t_o, sig_o, R_fit = ode['t'], ode['signal'], ode['R_fit']
w_o = build_full_weight_mask(t_o, sample['markers'], dmso_cal,
                             association_weight=0.0)
buf_o = (w_o > 0) & (t_o >= inj) & (t_o < rinse)

dmso = select_dmso_cal(sample['index'], dmso_cals)
c_func, _ = build_concentration_profile(dmso, sample['concentration_M'])
c_func_pulsed, _ = build_pulsed_concentration_profile(dmso, sample['concentration_M'])

creoptix_df = pd.read_csv("/mnt/c/Users/kgr26424/Documents/Creoptix_test/Data_release/2026-04(apr)-27 - EV712A data release V3.0.csv")  # Replace with the actual path to the CSV file containing Creoptix results
EXPERIMENT_DATE = f"{EXPERIMENT.split('_')[0][-2:]}/{EXPERIMENT.split('_')[0][-4:-2]}/{EXPERIMENT.split('_')[0][0:-4]}"  # Extract the date part from the experiment name
creoptix_df = creoptix_df[creoptix_df["Run date"] == EXPERIMENT_DATE]  # Filter data using the experiment date
creoptix_results = {
    'ka': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "ka (M-1s-1)"].iloc[0],
    'kd': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "kd (s-1)"].iloc[0],
    'KD': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "KD (M)"].iloc[0],
    'Rmax': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "Rmax (pg/mm2)"].iloc[0],
} # Creoptix results for the same compound, channel and experiment date.

simulate_creoptix = simulate_sensorgram(
    t=t_o,
    ka=creoptix_results['ka'],
    kd=creoptix_results['kd'],
    Rmax=creoptix_results['Rmax'],
    c_func=c_func_pulsed,
)

creoptix_residuals = sig_o - simulate_creoptix

simulate_dk = simulate_sensorgram(
    t=t_o,
    ka=dk['ka'],
    kd=dk['kd'],
    Rmax=dk['Rmax'],
    c_func=c_func,
)

dk_residuals = sig_o - simulate_dk

# Overlay all fits on the data
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
ax = axes[0]
ax.axvspan(t_o[0], inj,   alpha=0.08, color='gray')
ax.axvspan(inj,    rinse, alpha=0.10, color='orange')
for s_, e_ in _contiguous_spans(t_o, buf_o):
    ax.axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
    axes[1].axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
ax.axvspan(rinse, rend, alpha=0.18, color='mediumseagreen')
axes[1].axvspan(rinse, rend, alpha=0.18, color='mediumseagreen')

ax.plot(t_o, sig_o, 'k-', lw=0.75, alpha=0.6, label='Data')
ax.plot(t_o, R_fit, 'r-', lw=1.3,
        label=f'ODE fit  (KD = {ode["KD"]*1e6:.2f} µM)')
ax.plot(t_o, simulate_dk, ':', lw=1.3,
        label=f'DK fit  (KD = {dk["KD"]*1e6:.2f} µM)')
ax.plot(t_o, simulate_creoptix, '--', lw=1.3,
        label=f'Creoptix fit  (KD = {creoptix_results["KD"]*1e6:.2f} µM)')
ax.axhline(0, color='gray', ls=':', lw=0.5)
ax.set_ylabel('Signal (pg/mm²)')
ax.set_title(f'ODE, DK & Creoptix fit overlay — {sample["compound"]}')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)

resid = ode['residuals']
fitted_pts = w_o > 0
axes[1].plot(t_o[fitted_pts], resid[fitted_pts], 'r-', lw=0.5, label='ODE fit residuals')
axes[1].plot(t_o[fitted_pts], dk_residuals[fitted_pts], ':', lw=0.5, label='DK fit residuals')
axes[1].plot(t_o[fitted_pts], creoptix_residuals[fitted_pts], '--', lw=0.5, label='Creoptix fit residuals')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8, loc='best')

fig.tight_layout()
plt.show()

## 3. Non-convex optimisation: DK seeds vs random starting points

The 1:1 Langmuir model

$$\frac{dR}{dt} = k_a \, c(t) \, [R_{\max} - R(t)] - k_d \, R(t)$$

is **non-linear** in its parameters $(k_a, k_d, R_{\max})$. The least-squares
loss surface has a long, curved valley in the $(k_a, R_{\max})$ plane (their
product sets the steady-state response) plus shallow local minima where the
optimiser can stall. As a result, the kinetic estimates returned by an ODE
solver are sensitive to where the optimiser is started. The practical consequence
is that the kinetic estimates you get out of an ODE solver depend on where you
started it and that dependence is not something a user of the vendor software
gets to see directly.

`sensofit` defaults to **DK-seeded** starting points: the closed-form Direct
Kinetics solver provides initial guesses of $(k_a, k_d, R_{\max})$, and the
multi-start option then perturbs them log-normally
($\sigma = 0.5$ in log-space). This is fast and usually well-behaved.

To make the non-convexity visible, this section also runs the same fit from
**fully random log-uniform** starting points across the physical bounds used
internally by `ode_fit`:

**CHANGE LB AND UB ACCORDING TO VALUES IN CODE**

| parameter | lower bound | upper bound |
|-----------|------------:|------------:|
| $k_a$       | $10^{-1}$ M⁻¹ s⁻¹ | $10^{8}$ M⁻¹ s⁻¹ |
| $k_d$       | $10^{-6}$ s⁻¹     | $10^{1}$ s⁻¹     |
| $R_{\max}$  | $1$ pg/mm²        | $10^{4}$ pg/mm²  |

We compare the resulting probability distributions of $k_a$, $k_d$, $R_{\max}$
and $K_D$, and show why the **median** (not the mean) is the right summary
statistic when distributions are heavy-tailed.

In [ ]:
# --- Replicate the prep that ode_fit_sample does internally ----------------
# (DK initial estimates, pulsed c(t), full weight mask, trimmed window)
dk = dk_fit_sample(sample, dmso_cals, blanks=blanks)
t_full, signal_full = dk['t'], dk['signal']

dmso = select_dmso_cal(sample['index'], dmso_cals)
c_func, _ = build_pulsed_concentration_profile(dmso, sample['concentration_M'])

# Default association_weight=0 → fit on buffer pulses + dissociation only
w_full = build_full_weight_mask(t_full, sample['markers'], dmso, association_weight=0.0)
t_fit, sig_fit, w_fit, fit_mask = trim_to_fit_window(
    t_full, signal_full, w_full, sample['markers'])
print(f'DK initial estimates:')
print(f'  ka   = {dk["ka"]:.2e} M⁻¹ s⁻¹')
print(f'  kd   = {dk["kd"]:.4f} s⁻¹')
print(f'  Rmax = {dk["Rmax"]:.1f} pg/mm²')
print(f'  KD   = {dk["KD"]*1e6:.2f} µM')
print(f'\nFitting window: {len(t_fit)} pts, {w_fit.sum():.0f} weighted')

TO DO BEFORE RELEASE: REVIEW UPPER AND LOWER BOUNDS

In [ ]:
# --- ODE multi-start machinery ---------------------------------------------
# We use sensofit.ode_fitting._residuals_full directly, which is exactly the
# residual function `ode_fit_sample` uses inside its Phase-3 multi-start loop.
# This means the per-start parameters we collect below are drawn from the same
# distribution that `ode_fit_sample` would aggregate by median.

# ode_fit's physical bounds
LB = np.array([1e-1, 1e-9, 0.1])     # ka, kd, Rmax
UB = np.array([1e12,  1e1,  1e4])

def random_log_uniform_starts(n, rng):
    """Draw `n` random (ka, kd, Rmax) starts log-uniformly within [LB, UB]."""
    log_lo, log_hi = np.log10(LB), np.log10(UB)
    return [10 ** rng.uniform(log_lo, log_hi) for _ in range(n)]

def dk_perturbed_starts(n, rng, dk_seed, sigma=0.5):
    """Draw `n` log-normally perturbed starts around the DK seed (sensofit default)."""
    seed = np.array(dk_seed, dtype=float)
    return [np.clip(seed * np.exp(rng.normal(0, sigma, size=3)), LB, UB)
            for _ in range(n)]

def run_starts(starts, t, signal, c_func, w):
    """Run scipy.least_squares from each start using sensofit's residual."""
    rows = []
    for p0 in starts:
        try:
            opt = least_squares(
                _residuals_full, p0, args=(t, signal, c_func, w),
                bounds=(LB, UB), method='trf',
                ftol=1e-6, xtol=1e-6, gtol=1e-6, max_nfev=200, diff_step=1e-2)
            rows.append({
                'init_ka': p0[0], 'init_kd': p0[1], 'init_Rmax': p0[2],
                'ka': opt.x[0], 'kd': opt.x[1], 'Rmax': opt.x[2],
                'KD': opt.x[1] / opt.x[0],
                'cost': float(opt.cost), 'success': bool(opt.success),
                'nfev': int(opt.nfev),
            })
        except Exception:
            rows.append({
                'init_ka': p0[0], 'init_kd': p0[1], 'init_Rmax': p0[2],
                'ka': np.nan, 'kd': np.nan, 'Rmax': np.nan, 'KD': np.nan,
                'cost': np.nan, 'success': False, 'nfev': 0,
            })
    return pd.DataFrame(rows)

In [ ]:
# --- Run both strategies (this takes ~60–180 s) -----------------------------
N_STARTS = 10
RNG_SEED = 20260505
rng = np.random.default_rng(RNG_SEED)

dk_seed = (dk['ka'], dk['kd'], dk['Rmax'])
dk_results   = run_starts(dk_perturbed_starts(N_STARTS, rng, dk_seed),
                          t_fit, sig_fit, c_func, w_fit)
rand_results = run_starts(random_log_uniform_starts(N_STARTS, rng),
                          t_fit, sig_fit, c_func, w_fit)

dk_ok   = dk_results[dk_results['success']]
rand_ok = rand_results[rand_results['success']]
print(f'DK-perturbed:  {len(dk_ok)}/{N_STARTS} converged')
print(f'Random log-U:  {len(rand_ok)}/{N_STARTS} converged')

In [ ]:
# --- Histograms of ka, kd, Rmax, KD with median (solid) and mean (dashed) --
ode_single_start = ode_fit_sample(sample, dmso_cals, blanks=blanks, n_starts=1)

params = [('DK-perturbed', 'ka', r'$k_a$ (M⁻¹ s⁻¹)', True),
          ('Random log-U', 'ka', r'$k_a$ (M⁻¹ s⁻¹)', True),
          ('DK-perturbed', 'kd', r'$k_d$ (s⁻¹)',     True),
          ('Random log-U', 'kd', r'$k_d$ (s⁻¹)',     True),
          ('DK-perturbed', 'Rmax', r'$R_{\max}$ (pg/mm²)', False),
          ('Random log-U', 'Rmax', r'$R_{\max}$ (pg/mm²)', False),
          ('DK-perturbed', 'KD', r'$K_D$ (M)',        True),
          ('Random log-U', 'KD', r'$K_D$ (M)',        True)]

def _kde_pdf(vals, log_x, n=400):
    from scipy.stats import gaussian_kde
    """Return (x, density) for a Gaussian KDE. For log-scale axes we KDE in
    log-space and transform back so the density on a log axis looks natural."""
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals) & (vals > 0 if log_x else np.full_like(vals, True, dtype=bool))]
    if len(vals) < 2 or np.ptp(vals) == 0:
        return None, None
    if log_x:
        lv = np.log10(vals)
        kde = gaussian_kde(lv)
        pad =  2 * np.ptp(lv)
        x_log = np.linspace(lv.min() - pad, lv.max() + pad, n)
        return 10 ** x_log, kde(x_log)
    kde = gaussian_kde(vals)
    pad = 0.5 * np.ptp(vals)
    x = np.linspace(vals.min() - pad, vals.max() + pad, n)
    return x, kde(x)

fig, axes = plt.subplots(4, 2, figsize=(10, 15))
i=0
dk_lim = []
rand_lim = []
for ax, (test, col, label, log_x) in zip(axes.flat, params):
    if test == 'DK-perturbed':
        vals   = dk_ok[col].dropna().values
        color = 'steelblue'
    elif test == 'Random log-U':
        vals = rand_ok[col].dropna().values
        color = "darkorange"

    x, y = _kde_pdf(vals, log_x)
    
    if test == 'DK-perturbed':
        dk_lim.append([min(x), max(x)])
    if test == 'Random log-U':
        rand_lim.append([min(x), max(x)])
    if x is None:
        continue
    ax.fill_between(x, y, alpha=0.30, color=color)
    ax.plot(x, y, color=color, lw=1.4)
    ax.axvline(np.median(vals), color=color, lw=1.4, ls='-', label=f'Median = {np.median(vals):.3e}' if log_x else f'Median = {np.median(vals):.3f}')
    ax.axvline(np.mean(vals),   color=color, lw=1.0, ls='--', label=f'Mean = {np.mean(vals):.3e}' if log_x else f'Mean = {np.mean(vals):.3f}')
    ax.axvline(ode_single_start[col], color='black', lw=1.2, ls=':',
               label='ODE (1 start)')
    
    if test == 'Random log-U':
        dk_range = dk_lim[i][1] - dk_lim[i][0]
        rand_range = rand_lim[i][1] - rand_lim[i][0]
        range_ratio = rand_range / dk_range if dk_range > 0 else float('inf')
        range_str = f"{range_ratio:.1f}" if np.isfinite(range_ratio) else "inf"
        text = r"$\bf{Rand. range = "+range_str+"x DK range}$"
        ax.plot([], [], ' ', label=text)
        i+=1

    if log_x:
        ax.set_xscale("log")
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    ax.set_xlabel(f"{label} - {test} (n={len(vals)})")
    ax.set_ylabel('density')
    ax.legend(fontsize=6, loc='upper left' if test == "DK-perturbed" else 'upper right')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Parameter PDFs across {N_STARTS} starts — '
             f'{sample["compound"]} ({sample["concentration_M"]*1e6:.1f} µM)\n'
             'Solid line = median, dashed line = mean, dotted = published\n',
             fontsize=11)
fig.tight_layout()
plt.show()

The figure above illustrates the Probability density functions (PDF) of 
$k_a$, $k_d$, $R_{max}$ and $K_D$ across multi-start ODE fits, comparing
DK-perturbed seeds (blue) against fully random log-uniform starts (orange).

The random-start distributions are often visibly broader and bimodal, exactly
the local-minima problem made manifest, while the DK-perturbed distributions 
are generally tight because DK already sits in the right basin of attraction.
The qualitative point is that seeding matters, and that an unseen non-convex
landscape sits underneath any single reported $K_D$; it is the kind of thing
that makes us cautious about over-interpreting any one number, ours or the vendor's. 

In [ ]:
# --- Tabulate median / mean / IQR / convergence rate -----------------------
def _summary(df, name):
    out = {'strategy': name, 'converged_pct': 100 * df['success'].mean()}
    ok = df[df['success']]
    for col in ['ka', 'kd', 'Rmax', 'KD']:
        v = ok[col].dropna().values
        if len(v):
            q25, q75 = np.percentile(v, [25, 75])
            out[f'{col}_median'] = np.median(v)
            out[f'{col}_mean']   = np.mean(v)
            out[f'{col}_iqr']    = q75 - q25
    return out

summary = pd.DataFrame([
    _summary(dk_results,   'DK-perturbed'),
    _summary(rand_results, 'Random log-U'),
    {'strategy': 'ODE (1 start)', 'converged_pct': 100.0,
     'ka_median': ode_single_start['ka'],     'ka_mean': ode_single_start['ka'],     'ka_iqr': 0,
     'kd_median': ode_single_start['kd'],     'kd_mean': ode_single_start['kd'],     'kd_iqr': 0,
     'Rmax_median': ode_single_start['Rmax'], 'Rmax_mean': ode_single_start['Rmax'], 'Rmax_iqr': 0,
     'KD_median':   ode_single_start['KD'],   'KD_mean':   ode_single_start['KD'],   'KD_iqr':   0},
])
summary.set_index('strategy').T

**Discussion.** A few things to take away from the histograms above:

1. **Random starts often find more local minima.** The orange (random log-uniform)
   distributions are usually visibly broader and often bimodal — the optimiser ends
   up in a shallow local minimum from a poor starting point. The blue
   (DK-perturbed) distributions tend to be tighter because DK already sits in the
   right basin of attraction.
2. **The mean is misleading on heavy-tailed PDFs.** When even a handful of
   starts land in a bad local minimum (e.g.\ unrealistically large $k_a$
   compensated by tiny $R_{\max}$), the mean is pulled towards them. The
   median is robust.
3. **`sensofit` aggregates by median.** This is why `ode_fit` returns the
   median over converged multi-start fits, and why the published value
   (single-start, deterministic) sits very close to the **median** — not the
   mean — of the DK-perturbed distribution.

## 4. Residual weighting and goodness-of-fit

Pulsed waveform GCI sensorgrams contain three distinct regions: a baseline
before injection, an association phase made of alternating *analyte pulses*
(contaminated by RI bulk artefacts) and *buffer pulses* (clean binding
signal), and a dissociation phase after rinse-in. `sensofit` uses
`build_full_weight_mask` to weight residuals as

| region                     | default weight |
|----------------------------|---------------:|
| baseline                   | 0              |
| analyte pulses (RI)        | `association_weight` |
| buffer pulses              | 1              |
| dissociation (Rinse → end) | 1              |

The choice of `association_weight` ∈ [0, 1] is a trade-off:

- `0.0` (default) — fit on buffer pulses + dissociation only. Robust to RI
  artefacts but throws away kinetic information from the analyte pulses.
- `1.0` — fit everything. Squeezes all available data but is biased by RI
  spikes during the analyte pulses.

We sweep `association_weight` from 0 to 1 and report several
goodness-of-fit (GOF) metrics, **split by association vs dissociation
phase**, to show how the optimum shifts.

In [ ]:
# --- GOF helpers (chi^2, R^2, RMSE from cedric-dev; AIC/BIC added) --------
def _align(exp, fit):
    n = min(len(exp), len(fit))
    return np.asarray(exp[:n]), np.asarray(fit[:n])

def chi2(exp, fit, dof=3):
    exp, fit = _align(exp, fit)
    sigma = np.std(exp)
    return float(np.sum(((exp - fit) / sigma) ** 2) / dof)

def r_squared(exp, fit):
    exp, fit = _align(exp, fit)
    ss_res = np.sum((exp - fit) ** 2)
    ss_tot = np.sum((exp - np.mean(exp)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

def rmse(exp, fit, p=3):
    exp, fit = _align(exp, fit)
    ss_res = np.sum((exp - fit) ** 2)
    return float(np.sqrt(ss_res / max(len(exp) - p, 1)))

def aic(exp, fit, k=3):
    exp, fit = _align(exp, fit)
    n = len(exp); rss = float(np.sum((exp - fit) ** 2))
    if rss <= 0 or n <= 0:
        return float('nan')
    return n * np.log(rss / n) + 2 * k

def bic(exp, fit, k=3):
    exp, fit = _align(exp, fit)
    n = len(exp); rss = float(np.sum((exp - fit) ** 2))
    if rss <= 0 or n <= 0:
        return float('nan')
    return n * np.log(rss / n) + k * np.log(n)

In [ ]:
# --- Sweep association_weight and collect kinetic params + per-phase GOF --
def weight_sweep(sample, dmso_cals, blanks, weights):
    rinse     = sample['markers'].get('Rinse',    0.0)
    rinse_end = sample['markers'].get('RinseEnd', np.inf)
    rows = []
    for w in weights:
        ode = ode_fit_sample(sample, dmso_cals, blanks=blanks,
                             association_weight=w, n_starts=1)
        t_o   = ode['t']
        sig_o = ode['signal']
        R_fit = ode['R_fit']
        valid = ~np.isnan(R_fit)
        diss  = valid & (t_o >= rinse) & (t_o <= rinse_end)
        asso  = valid & ~diss
        row = {
            'weight': w,
            'ka': ode['ka'], 'ka_se': ode.get('ka_se', np.nan),
            'kd': ode['kd'], 'kd_se': ode.get('kd_se', np.nan),
            'Rmax': ode['Rmax'], 'Rmax_se': ode.get('Rmax_se', np.nan),
            'KD': ode['KD'],
            'chi2_total': chi2(sig_o[valid], R_fit[valid]),
            'chi2_asso':  chi2(sig_o[asso],  R_fit[asso])  if asso.sum() else np.nan,
            'chi2_diss':  chi2(sig_o[diss],  R_fit[diss])  if diss.sum() else np.nan,
            'rmse_total': rmse(sig_o[valid], R_fit[valid]),
            'rmse_asso':  rmse(sig_o[asso],  R_fit[asso])  if asso.sum() else np.nan,
            'rmse_diss':  rmse(sig_o[diss],  R_fit[diss])  if diss.sum() else np.nan,
            'r2_total':   r_squared(sig_o[valid], R_fit[valid]),
            'aic':        aic(sig_o[valid], R_fit[valid]),
            'bic':        bic(sig_o[valid], R_fit[valid]),
        }
        rows.append(row)
    return pd.DataFrame(rows)

WEIGHTS = np.round(np.linspace(0, 1, 11), 2)
ws_df = weight_sweep(sample, dmso_cals, blanks, weights=WEIGHTS)
ws_df.round(4)

In [ ]:
# --- Two-row figure: kinetic params (top) + per-phase GOF (bottom) --------
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1 — parameters with SE error bars
for ax, col, ylabel, log_y in zip(
        axes[0],
        ['ka', 'kd', 'Rmax', 'KD'],
        [r'$k_a$ (M⁻¹ s⁻¹)', r'$k_d$ (s⁻¹)', r'$R_{\max}$ (pg/mm²)', r'$K_D$ (M)'],
        [True, True, False, True]):
    se_col = f'{col}_se' if f'{col}_se' in ws_df.columns else None
    yerr = ws_df[se_col] if se_col else None
    ax.errorbar(ws_df['weight'], ws_df[col], yerr=yerr,
                marker='o', color='steelblue', capsize=3)
    if log_y:
        ax.set_yscale('log')
    ax.set_xlabel('association_weight', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Row 2 — GOF metrics, per-phase
for ax, metric, ylabel in zip(
        axes[1],
        ['chi2', 'rmse', 'r2', 'aic'],
        ['χ² (per dof)', 'RMSE (pg/mm²)', 'R²', 'AIC']):
    if metric == 'r2':
        ax.plot(ws_df['weight'], ws_df['r2_total'], 'k-o', label='total')
    elif metric == 'aic':
        ax.plot(ws_df['weight'], ws_df['aic'], 'k-o', label='AIC')
        ax.plot(ws_df['weight'], ws_df['bic'], 'r-s', label='BIC')
    else:
        ax.plot(ws_df['weight'], ws_df[f'{metric}_total'], 'k-o', label='total')
        ax.plot(ws_df['weight'], ws_df[f'{metric}_asso'],  'b--^', label='association')
        ax.plot(ws_df['weight'], ws_df[f'{metric}_diss'],  'g--v', label='dissociation')
    ax.set_xlabel('association_weight', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Weight sensitivity — {sample["compound"]} '
             f'({sample["concentration_M"]*1e6:.1f} µM)', fontsize=12)
fig.tight_layout()
plt.show()

The figure above shows the impact of `association_weight` on the four kinetic estimates ($k_a$, $k_d$, $R_{max}$, $K_D$, top row) and on per-phase goodness-of-fit (GOF) metrics (χ², RMSE, R², AIC/BIC, bottom row). Error bars on the parameters are SensoFit's reported standard errors; GOF metrics are split into association, dissociation, and total.

REMARK: I AM NOT 100% CONFIDENT WITH THE WAY CHI2 IS DEFINED IN THE NOTEBOOK (The noise (sigma) is taken from the std of the whole experiments, but ideally the noise should only be measured from the std of the baseline (I tried to change but then I got chi2 values that didn't make sense).)

In [ ]:
# --- Heatmap of z-scored GOF metrics vs weight (matplotlib only) ---------
metric_cols = ['chi2_total', 'chi2_asso', 'chi2_diss',
               'rmse_total', 'rmse_asso', 'rmse_diss',
               'aic', 'bic']
heat = ws_df[metric_cols].copy()
# Z-score each metric so they share a colour scale (lower = better, dark = better)
heat_z = (heat - heat.mean()) / heat.std(ddof=0)
Z = heat_z.T.values            # rows = metrics, cols = weights
A = heat.T.round(2).values     # raw annotations

fig, ax = plt.subplots(figsize=(10, 5))
vmax = float(np.nanmax(np.abs(Z)))
im = ax.imshow(Z, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(ws_df)))
ax.set_xticklabels([f'{w:.1f}' for w in ws_df['weight']])
ax.set_yticks(range(len(metric_cols)))
ax.set_yticklabels(metric_cols)

for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        ax.text(j, i, f'{A[i, j]:g}', ha='center', va='center',
                fontsize=8,
                color='white' if abs(Z[i, j]) > 0.6 * vmax else 'black')
fig.colorbar(im, ax=ax, label='z-score (lower = better)')
ax.set_title(f'Goodness-of-fit vs association_weight — {sample["compound"]}')
ax.set_xlabel('association_weight')
ax.set_ylabel('metric')
fig.tight_layout()
plt.show()

The figure above shows the heatmap of z-scored goodness-of-fit metrics (χ² total / association / dissociation, RMSE total / association / dissociation, AIC, BIC) versus `association_weight`. Colour is z-score (lower / darker blue = better); cell annotations are the raw metric values.

**Discussion.** The sweep makes a few patterns clear:

- **Pure dissociation fitting (`association weight = 0`)** has the lowest dissociation
  RMSE by construction, but pays for it with a worse association RMSE
  and noisier $R_{\max}$ (because $R_{\max}$ is anchored only by where the
  curve plateaus during the buffer pulses).
- **Full-weight fitting (`association weight = 1`)** achieves the best total χ²/RMSE
  but biases the parameters: the analyte-pulse RI spikes get folded into
  the residual and the optimiser distorts $k_a$ to absorb them.

The published kinetics in this release use the `sensofit` defaults
(`association_weight = 0.0`, dissociation-only fitting), however this is something
we'd love to get more input from the community about. 

**NEW CONTENT FOR THE NOTEBOOK, PLEASE REVIEW BEFORE RELEASE**

## 6. The case of negative steady-state values after last dissociation

One of the issues that we face for some of the fittings is when the steady-state
after the last dissociation (which can also be referred to as the return to baseline)
reaches negative values. Because of the way the 1:1 Langmuir ODE is defined, negative
values are mathematically and physically impossible. The solution decays and plateaus
asymptoticallytoward zero, and therefore never crosses it.

This section illustrates such a case, and shows how applying a simple baseline correction
to the data can impact the value of the predicted KD. 

In [ ]:
sample_id = "ASAP-0038628" # Other IDs from the same experiment that can be used for testing: "ASAP-0044171", "ASAP-0039722", "ASAP-0044334", "ASAP-0039823"
sample = next(s for s in samples if sample_id in s['compound'])
print(f'Sample with negative steady-state: {sample["compound"]}  '
      f'({sample["concentration_M"]*1e6:.1f} µM, cycle {sample["index"]})')

t_raw   = sample['time']
sig_dr, blank_idx = double_reference(sample, blanks)
inj   = sample['markers'].get('Injection', t_raw[0])
rinse = sample['markers'].get('Rinse',     t_raw[-1])
rend  = sample['markers'].get('RinseEnd',  t_raw[-1])
neg_value = min(sig_dr[t_raw > rinse])

dmso_cal = select_dmso_cal(sample['index'], dmso_cals)
w_full   = build_full_weight_mask(t_raw, sample['markers'], dmso_cal,
                                  association_weight=0.0)
buf_mask = (w_full > 0) & (t_raw >= inj) & (t_raw < rinse)

fig, ax = plt.subplots(figsize=(12, 4.5))
# Phase shading
ax.axvspan(t_raw[0], inj,   alpha=0.08, color='gray',   label='Baseline (not fitted)')
ax.axvspan(inj,      rinse, alpha=0.10, color='orange', label='Association — analyte pulses (RI)')
for s_, e_ in _contiguous_spans(t_raw, buf_mask):
    ax.axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
ax.axvspan(rinse,    rend,  alpha=0.18, color='mediumseagreen',
           label='Dissociation (fitted)')
ax.fill_between([], [], alpha=0.18, color='cornflowerblue',
                label='Association — buffer pulses (fitted)')

ax.plot(t_raw, sig_dr, 'k-', lw=0.6, label=f'Double-referenced data (blank {blank_idx})')
ax.plot([], [], ' ', label=f'Min. value in dissociation: {neg_value:.2f} pg/mm²')
ax.axhline(0, color='gray', ls=':', lw=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Signal (pg/mm²)')
ax.set_title(f'{sample["compound"]} — {sample["concentration_M"]*1e6:.1f} µM '
             f'(cycle {sample["index"]})')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
n_starts = 3
ode = ode_fit_sample(sample, dmso_cals, blanks=blanks, n_starts=n_starts)
ode_corr = ode_fit_sample(sample, dmso_cals, blanks=blanks, neg_ss_correction=True, n_starts=n_starts)

print(f'=== ODE refinement ({n_starts} starts, NO correction) ===')
print(f'  ka   = {ode["ka"]:.2e} ± {ode.get("ka_se", float("nan")):.2e}')
print(f'  kd   = {ode["kd"]:.4f}  ± {ode.get("kd_se", float("nan")):.4f}')
print(f'  Rmax = {ode["Rmax"]:.1f}   ± {ode.get("Rmax_se", float("nan")):.2f}')
print(f'  KD   = {ode["KD"]*1e6:.2f} µM')
print(f'  σ    = {ode["sigma_residual"]:.3f} pg/mm²   ({ode["message"]})')
print()
print(f'=== ODE refinement ({n_starts} starts, WITH correction) ===')
print(f'  ka   = {ode_corr["ka"]:.2e} ± {ode_corr.get("ka_se", float("nan")):.2e}')
print(f'  kd   = {ode_corr["kd"]:.4f}  ± {ode_corr.get("kd_se", float("nan")):.4f}')
print(f'  Rmax = {ode_corr["Rmax"]:.1f}   ± {ode_corr.get("Rmax_se", float("nan")):.2f}')
print(f'  KD   = {ode_corr["KD"]*1e6:.2f} µM')
print(f'  σ    = {ode_corr["sigma_residual"]:.3f} pg/mm²   ({ode_corr["message"]})')

In [ ]:
t_o, sig_o, R_fit = ode['t'], ode['signal'], ode['R_fit']
t_o_corr, sig_o_corr, R_fit_corr = ode_corr['t'], ode_corr['signal'], ode_corr['R_fit']

dmso = select_dmso_cal(sample['index'], dmso_cals)
c_func_pulsed, _ = build_pulsed_concentration_profile(dmso, sample['concentration_M'])

creoptix_results = {
    'ka': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "ka (M-1s-1)"].iloc[0],
    'kd': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "kd (s-1)"].iloc[0],
    'KD': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "KD (M)"].iloc[0],
    'Rmax': creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == "Ch 2-1"), "Rmax (pg/mm2)"].iloc[0],
} # Creoptix results for the same compound, channel and experiment date.

simulate_creoptix = simulate_sensorgram(
    t=t_o,
    ka=creoptix_results['ka'],
    kd=creoptix_results['kd'],
    Rmax=creoptix_results['Rmax'],
    c_func=c_func_pulsed,
)

creoptix_residuals = sig_o - simulate_creoptix

# Overlay all fits on the data
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
ax = axes[0]
ax.axvspan(t_o[0], inj,   alpha=0.08, color='gray')
ax.axvspan(inj,    rinse, alpha=0.10, color='orange')
for s_, e_ in _contiguous_spans(t_o, buf_o):
    ax.axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
    axes[1].axvspan(s_, e_, alpha=0.18, color='cornflowerblue')
ax.axvspan(rinse, rend, alpha=0.18, color='mediumseagreen')
axes[1].axvspan(rinse, rend, alpha=0.18, color='mediumseagreen')

ax.plot(t_o, sig_o, 'k-', lw=1, alpha=0.6, label='Data')
ax.plot(t_o_corr, sig_o_corr, 'k--', lw=0.75, alpha=0.6, label='Data (with neg. ss. correction)')
ax.plot(t_o, R_fit, 'r-', lw=1.3,
        label=f'ODE fit  (KD = {ode["KD"]*1e6:.2f} µM)')
ax.plot(t_o_corr, R_fit_corr, 'r--', lw=1,
        label=f'ODE fit (with neg. ss. correction)  (KD = {ode_corr["KD"]*1e6:.2f} µM)')
ax.plot(t_o, simulate_creoptix, '--', lw=1.3,
        label=f'Creoptix fit  (KD = {creoptix_results["KD"]*1e6:.2f} µM)')
ax.axhline(0, color='gray', ls=':', lw=0.5)
ax.set_ylabel('Signal (pg/mm²)')
ax.set_title(f'ODE, DK & Creoptix fit overlay — {sample["compound"]}')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)

resid = ode['residuals']
fitted_pts = w_o > 0
axes[1].plot(t_o[fitted_pts], resid[fitted_pts], 'r-', lw=0.5, label='ODE fit residuals')
axes[1].plot(t_o_corr[fitted_pts], R_fit_corr[fitted_pts] - sig_o_corr[fitted_pts], 'r--', lw=0.5, label='ODE fit (with neg. ss. correction) residuals')
axes[1].plot(t_o[fitted_pts], creoptix_residuals[fitted_pts], '--', lw=0.5, label='Creoptix fit residuals')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8, loc='best')

fig.tight_layout()
plt.show()

Depending on the sample ID you choose to illustrates this challenge, you can see a more or less big
difference in the estimated $K_D$ with or without the correction. For one of the samples (ID: ASAP-0038628)
the estimated $K_D$ without correction is "off" by $10^2$ (order of 2), which shows how a "simple" correction
can have a significant impact on the estimation.

## 7. Brief comparison with Creoptix evaluations

As mentionned in the blog post and ealier in this notebook, `SensoFit`'s pupose was primarily to be open about the analysis
of the sensorgram. We believe that, if you have read up until here, you should be convince that evaluating the sensorgram and
estimating the "right" $K_D$ is not a simple task: many things can influence the fitting and output one interpretation over the
other. However, `SensoFit` ambition for the future is also to be a robust tool that can be used to estimate accurately affinty
measurements in a high throughput fashion. This section goes through a subset of the samples and compare the `SensoFit` estimations
with the published one from Creoptix.

In [ ]:
N_SAMPLES = 50 # Number of ODE fits to run and compare with Creoptix results
N_STARTS = 3  # Number of starts for each ODE fit (set to 1 for faster execution during testing)
channel_dict_conversion = {
    "FC2-FC1": "Ch 2-1",
    "FC3-FC1": "Ch 3-1",
    "FC4-FC1": "Ch 4-1",
}
comp_data = {"Cycle": [],
             "Channel": [],
             "Compound": [],
             "KD SensoFit (M)": [],
             "KD Creoptix (M)": [],
             }
for i in range(N_SAMPLES):
    sample = samples[i]
    if sample["index"] == creoptix_df["Cycle number"].values[i] and sample["compound"] == creoptix_df["ASAP IDs"].values[i] and channel_dict_conversion[sample["channel"]] == creoptix_df["Channel"].values[i]:
        print(f'\n=== Sample {i+1}/{N_SAMPLES}: {sample["compound"]} ({sample["concentration_M"]*1e6:.1f} µM) ===')
        ode = ode_fit_sample(sample, dmso_cals, blanks=blanks, n_starts=N_STARTS)
        comp_data["Cycle"].append(sample["index"])
        comp_data["Channel"].append(channel_dict_conversion[sample["channel"]])
        comp_data["Compound"].append(sample["compound"])
        comp_data["KD SensoFit (M)"].append(ode["KD"])
        comp_data["KD Creoptix (M)"].append(creoptix_df.loc[(creoptix_df["ASAP IDs"] == sample["compound"]) & (creoptix_df["Cycle number"] == sample["index"]) & (creoptix_df["Channel"] == channel_dict_conversion[sample["channel"]]), "KD (M)"].iloc[0])
    else:
        print(f'\n=== Sample {i+1}/{N_SAMPLES}: {sample["compound"]} ({sample["concentration_M"]*1e6:.1f} µM) - Skipped (no matching Creoptix data) ===')

comp_df = pd.DataFrame(comp_data)
comp_df.head()

In [ ]:
import seaborn as sns
from scipy.stats import linregress

res = linregress(comp_df["KD Creoptix (M)"], comp_df["KD SensoFit (M)"])

slope = res.slope
intercept = res.intercept
r_value = res.rvalue
r_squared = r_value**2
p_value = res.pvalue

ax = sns.regplot(
    data=comp_df,
    x="KD Creoptix (M)",
    y="KD SensoFit (M)",
    ci=95,
    scatter_kws={"s": 50},
    line_kws={"label": f"$R^2$ = {r_squared:.2f}\nSlope = {slope:.2f}"}
)

ax.legend()

ax.set_title("KD Creoptix vs KD SensoFit")
ax.set_xlabel("KD Creoptix (M)")
ax.set_ylabel("KD SensoFit (M)")

plt.show()

In [ ]:
# Removing outliers and re-plotting
Q1 = comp_df[["KD Creoptix (M)", "KD SensoFit (M)"]].quantile(0.25)
Q3 = comp_df[["KD Creoptix (M)", "KD SensoFit (M)"]].quantile(0.75)
IQR = Q3 - Q1

comp_df_iqr = comp_df[
    ~(
        (comp_df[["KD Creoptix (M)", "KD SensoFit (M)"]] < (Q1 - 1.5 * IQR)) |
        (comp_df[["KD Creoptix (M)", "KD SensoFit (M)"]] > (Q3 + 1.5 * IQR))
    ).any(axis=1)
]

res = linregress(comp_df_iqr["KD Creoptix (M)"], comp_df_iqr["KD SensoFit (M)"])

slope = res.slope
intercept = res.intercept
r_value = res.rvalue
r_squared = r_value**2
p_value = res.pvalue

ax = sns.regplot(
    data=comp_df_iqr,
    x="KD Creoptix (M)",
    y="KD SensoFit (M)",
    ci=95,
    scatter_kws={"s": 50},
    line_kws={"label": f"$R^2$ = {r_squared:.2f}\nSlope = {slope:.2f}"}
)

ax.legend()

ax.set_title(f"KD Creoptix vs KD SensoFit\n"
             f"(outliers removed, n={len(comp_df)-len(comp_df_iqr)})")
ax.set_xlabel("KD Creoptix (M)")
ax.set_ylabel("KD SensoFit (M)")

plt.show()

## Reproducibility

- Random seed for the multi-start study: `RNG_SEED = 20260505`
  (change this and re-run section 3 to convince yourself the broad shape
  of the histograms is seed-independent)
- The multi-start helpers (`random_log_uniform_starts`, `dk_perturbed_starts`,
  `run_starts`) call `sensofit.ode_fitting._residuals_full` directly, so the
  per-start parameters they collect are drawn from exactly the same
  distribution that `ode_fit_sample` aggregates by median.
- All goodness-of-fit metrics (`chi2`, `r_squared`, `rmse`, `aic`, `bic`)
  and the `weight_sweep` driver are defined inline in this notebook.